# 🧪 RAG Hands-on (Live Session) — Module 2 · Lecture 4

We build a minimal RAG assistant for a fictional company, **Acme Corp**, in 8 small steps:

**documents → chunks → embeddings → vector store → hybrid search → rerank → LLM answer with citations**

| Step |
|---|
| 0 · Setup |
| 1 · Knowledge base |
| 2 · Sparse vs dense: why embeddings? |
| 3 · Chunking |
| 4 · Vector store: FAISS |
| 5 · Vector DB: Chroma + metadata filter |
| 6 · Hybrid search (BM25 + dense, RRF) |
| 7 · Rerank with a cross-encoder |
| 8 · Generate: the full `ask()` pipeline |
| Wrap-up |

## 0 · Setup

**Objective:** Set up the Python environment, download required libraries, configure the hardware accelerator (CPU/GPU), and initialize our target LLM and embedding models.

In [ ]:
%%capture
!pip install -q faiss-cpu chromadb rank_bm25 langchain-text-splitters

In [ ]:
import re, numpy as np, torch, warnings
warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"     # bi-encoder, 384-dim
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"        # cross-encoder reranker
LLM_MODEL    = "Qwen/Qwen2.5-1.5B-Instruct" if DEVICE == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct"
print(DEVICE, "|", LLM_MODEL)

cuda | Qwen/Qwen2.5-1.5B-Instruct


### 📊 Output Significance
By setting up our models and verifying our hardware accelerator (CPU or GPU), we establish a consistent execution baseline. This guarantees that model inference and text vectorization will run optimally in subsequent steps.

## 1 · Knowledge base

**Objective:** Define raw, unstructured documents representing various company policies (HR, IT, facilities) to act as the source of truth for our assistant.

In [ ]:
DOCS = {
"hr_policy_india.md": """# Acme-India HR Policy
## Annual leave
Employees accrue 18 days of paid leave annually. Unused leave of up to 30 days carries over into the next year.
## Sick leave
Employees get 12 days of sick leave per year. A medical certificate is needed for absences over 3 days.
## Remote work
Employees may work from home up to 2 days per week with their manager's approval.""",

"hr_policy_us.md": """# Acme-US HR Policy
## Annual leave
Employees receive 15 days of paid time off (PTO) per year. PTO does not carry over.
## Sick leave
Employees get 10 days of sick leave per year.
## Remote work
Employees may work from home up to 3 days per week.""",

"it_helpdesk.md": """# Acme IT Helpdesk FAQ
## Error E1047: login token expired
Appears when your single sign-on (SSO) session has expired. Fix: clear cookies for sso.acme.com and log in again.
## Error E2001: disk quota exceeded
Your network home drive has reached its 50 GB limit. Delete old files or request more space on the IT portal.
## Password reset
Passwords expire every 90 days. Reset yours at password.acme.com.""",

"facilities.md": """# Acme Facilities Guide
## Cafeteria
The cafeteria is open from 8 am to 6 pm on working days. Lunch is subsidised.
## Gym
The office gym on the 3rd floor is open from 6 am to 10 pm.""",
}

### 📊 Output Significance
Defining our raw knowledge base programmatically ensures that our fictional enterprise policies (covering HR, IT, and facilities) are fully loaded and prepared in memory. These raw text strings are ready to be structured and queried.

## 2 · Sparse vs dense: why embeddings?

**Objective:** Compare keyword-based matching (BM25) with vector-based semantic matching (embeddings) when matching queries with structurally different but semantically identical words.

In [ ]:
from rank_bm25 import BM25Okapi

docs = ["Employees accrue 18 days of paid leave annually.",
        "Unused leave up to 30 days carries over to next year.",
        "The cafeteria is open from 8 am to 6 pm."]
query = "How much vacation do I get?"

tokenize = lambda t: re.findall(r"[a-z0-9]+", t.lower())
bm25 = BM25Okapi([tokenize(d) for d in docs])
print(bm25.get_scores(tokenize(query)))      # all zeros: no shared words!

[0. 0. 0.]


In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
doc_vecs = embedder.encode(docs, normalize_embeddings=True)    # (3, 384), unit length
q_vec    = embedder.encode(query, normalize_embeddings=True)

print((doc_vecs @ q_vec).round(2))           # normalised → dot product = cosine similarity

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[0.44 0.42 0.02]


✅ Dense embeddings match **meaning** ("vacation" ≈ "paid leave") even with zero shared words.

### 📊 Output Significance
The mathematical output demonstrates that traditional exact keyword indexing (like BM25) fails on semantic equivalents with no exact keyword overlap (giving scores of `0`). In contrast, dense embeddings calculate high similarity scores, showing they capture the conceptual meaning of natural language queries.

## 3 · Chunking

**Objective:** Split the long, raw policy documents into smaller, coherent chunks based on headers, while preserving metadata (like document source and company context).

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

splitter = MarkdownHeaderTextSplitter([("#", "title"), ("##", "section")])
company_of = {"hr_policy_india.md": "Acme-India", "hr_policy_us.md": "Acme-US"}

chunks = []
for source, text in DOCS.items():
    for piece in splitter.split_text(text):
        m = piece.metadata
        chunks.append({
            "text": f"{m['title']} › {m['section']}: {piece.page_content}",
            "meta": {"source": source, "section": m["section"],
                     "company": company_of.get(source, "Acme (all)")},
        })

print(len(chunks), "chunks")
for c in chunks[:3]:
    print(c["meta"], "\n  ", c["text"])

11 chunks
{'source': 'hr_policy_india.md', 'section': 'Annual leave', 'company': 'Acme-India'} 
   Acme-India HR Policy › Annual leave: Employees accrue 18 days of paid leave annually. Unused leave of up to 30 days carries over into the next year.
{'source': 'hr_policy_india.md', 'section': 'Sick leave', 'company': 'Acme-India'} 
   Acme-India HR Policy › Sick leave: Employees get 12 days of sick leave per year. A medical certificate is needed for absences over 3 days.
{'source': 'hr_policy_india.md', 'section': 'Remote work', 'company': 'Acme-India'} 
   Acme-India HR Policy › Remote work: Employees may work from home up to 2 days per week with their manager's approval.


### 📊 Output Significance
Dividing our long documents into small, logical, and context-preserving text blocks ensures they will fit perfectly within context constraints during retrieval. It also allows the system to reference specific, scoped policies with high accuracy.

## 4 · Vector store: FAISS

**Objective:** Convert the text chunks into continuous vectors using an embedding model and index them inside a FAISS store for rapid nearest-neighbor similarity searches.

In [ ]:
import faiss

chunk_vecs = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True).astype("float32")
index = faiss.IndexFlatIP(chunk_vecs.shape[1])
index.add(chunk_vecs)

def dense_search(question, k=3):
    qv = embedder.encode([question], normalize_embeddings=True).astype("float32")  # SAME model
    scores, ids = index.search(qv, k)
    return [int(i) for i in ids[0]], scores[0]

ids, scores = dense_search("How much vacation do I get?")
for i, s in zip(ids, scores):
    print(f"{s:.3f}  {chunks[i]['text'][:80]}")

0.407  Acme-US HR Policy › Sick leave: Employees get 10 days of sick leave per year.
0.369  Acme-US HR Policy › Annual leave: Employees receive 15 days of paid time off (PT
0.340  Acme-India HR Policy › Annual leave: Employees accrue 18 days of paid leave annu


⚠️ FAISS returns only **ids and scores**. *We* keep `chunks` to map ids back to text. Also: both India and US leave policies come back. Which is right depends on who asks → metadata filtering.

### 📊 Output Significance
FAISS delivers extremely fast index search retrieval based on mathematical dot-products. This returns relevant candidate IDs with distance scores, confirming that our database is indexable and can identify semantic neighbors in milliseconds.

## 5 · Vector DB: Chroma + metadata filter

**Objective:** Instantiate a lightweight vector database (Chroma) that pairs each document's embedding vector alongside its text and metadata tags, allowing for constrained searches.

In [ ]:
import chromadb

client = chromadb.EphemeralClient()
try: client.delete_collection("acme")          # safe to re-run
except Exception: pass
col = client.create_collection("acme", metadata={"hnsw:space": "cosine"})   # match the model's metric

col.add(ids=[str(i) for i in range(len(chunks))],
        embeddings=chunk_vecs,
        documents=[c["text"] for c in chunks],
        metadatas=[c["meta"] for c in chunks])

In [ ]:
q = "How many days of leave do I get?"
qv = embedder.encode([q], normalize_embeddings=True)

for where in [None, {"company": "Acme-US"}]:
    res = col.query(query_embeddings=qv, n_results=2, where=where)
    print("filter:", where)
    for doc, dist in zip(res["documents"][0], res["distances"][0]):
        print(f"   cos={1 - dist:.2f}  {doc[:75]}")     # Chroma returns distance = 1 - cosine

filter: None
   cos=0.58  Acme-India HR Policy › Annual leave: Employees accrue 18 days of paid leave
   cos=0.54  Acme-US HR Policy › Sick leave: Employees get 10 days of sick leave per yea
filter: {'company': 'Acme-US'}
   cos=0.54  Acme-US HR Policy › Sick leave: Employees get 10 days of sick leave per yea
   cos=0.51  Acme-US HR Policy › Annual leave: Employees receive 15 days of paid time of


### 📊 Output Significance
Integrating Chroma DB allows us to pair vectors alongside structured metadata fields. The queried output demonstrates successful metadata constraint execution: filtering queries specifically for 'Acme-US' context ensures that irrelevant geographic records (e.g., Indian policies) are completely filtered out.

## 6 · Hybrid search: BM25 + dense, fused with RRF

**Objective:** Combine the high precision of keyword search (for exact matches like IDs and error codes) with the semantic capabilities of dense search, fusing their rankings using Reciprocal Rank Fusion (RRF).

In [ ]:
bm25_kb = BM25Okapi([tokenize(c["text"]) for c in chunks])

def bm25_search(question, k=5):
    return [int(i) for i in np.argsort(-bm25_kb.get_scores(tokenize(question)))[:k]]

def rrf(rankings, k=60):
    fused = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            fused[doc] = fused.get(doc, 0) + 1 / (k + rank)
    return sorted(fused, key=fused.get, reverse=True)

def hybrid_search(question, k=5):
    return rrf([bm25_search(question, k), dense_search(question, k)[0]])[:k]

for q in ["error E1047 when I log in", "When does the canteen open?"]:
    print(q)
    for name, fn in [("BM25", bm25_search), ("Dense", lambda x: dense_search(x)[0]), ("Hybrid", hybrid_search)]:
        print(f"  {name:<6}", [chunks[i]["meta"]["section"][:20] for i in fn(q)[:3]])

error E1047 when I log in
  BM25   ['Error E1047: login t', 'Error E2001: disk qu', 'Annual leave']
  Dense  ['Error E1047: login t', 'Error E2001: disk qu', 'Password reset']
  Hybrid ['Error E1047: login t', 'Error E2001: disk qu', 'Annual leave']
When does the canteen open?
  BM25   ['Gym', 'Annual leave', 'Cafeteria']
  Dense  ['Cafeteria', 'Gym', 'Password reset']
  Hybrid ['Gym', 'Cafeteria', 'Annual leave']


### 📊 Output Significance
This comparison highlights the benefits of hybrid search. By fusing BM25 (exact keyword match) and Dense (semantic match) rankings using Reciprocal Rank Fusion, the resulting top candidate list safely handles both technically specific search keywords (such as code identifiers) and natural conceptual references.

## 7 · Rerank with a cross-encoder
Bi-encoder: fast, retrieves a shortlist. Cross-encoder: reads (question, chunk) **together**: slower but more accurate, so we only use it on the shortlist.

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder(RERANK_MODEL, device=DEVICE)

def rerank(question, candidate_ids, top_n=3):
    scores = reranker.predict([(question, chunks[i]["text"]) for i in candidate_ids])
    return [candidate_ids[j] for j in np.argsort(-scores)[:top_n]]

q = "How often do I have to change my password?"
print([chunks[i]["meta"]["section"] for i in rerank(q, hybrid_search(q, k=8))])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

['Password reset', 'Remote work', 'Remote work']


### 📊 Output Significance
The output shows how a lightweight, fast bi-encoder/hybrid search can safely gather broad candidates, while a slower but highly precise cross-encoder reorders them. This puts the most semantically relevant text chunk at index rank `1`, which is ideal for LLM generation.

## 8 · Generate: the full pipeline

**Objective:** Build the complete end-to-end RAG workflow (`ask()` function) that retrieves, filters, reranks, and prompts the local instruction-tuned LLM to write a concise response cited with source references.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

gen_tok = AutoTokenizer.from_pretrained(LLM_MODEL)
gen_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL).to(DEVICE).eval()

@torch.no_grad()
def generate(messages, max_new_tokens=200):
    prompt = gen_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = gen_tok(prompt, return_tensors="pt").to(DEVICE)
    out = gen_model.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                             max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=gen_tok.eos_token_id)
    return gen_tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
SYSTEM = ("Answer using ONLY the numbered context. Cite passages like [1]. "
          "If the answer is not in the context, say: I don't know based on the Acme documents.")

def ask(question, company=None):
    cands = hybrid_search(question, k=8)                                   # 1. retrieve
    if company:                                                            #    + metadata filter
        cands = [i for i in cands if chunks[i]["meta"]["company"] in (company, "Acme (all)")]
    top = rerank(question, cands)                                          # 2. rerank
    context = "\n".join(f"[{n}] {chunks[i]['text']}" for n, i in enumerate(top, 1))
    answer = generate([{"role": "system", "content": SYSTEM},              # 3. augment + generate
                       {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}])
    print("❓", question, f"({company})" if company else "")
    print("💬", answer)
    print("📚", [chunks[i]["meta"]["source"] + " › " + chunks[i]["meta"]["section"] for i in top], "\n")

ask("How much vacation do I get?", company="Acme-India")
ask("How much vacation do I get?", company="Acme-US")
ask("I'm getting error E1047, what should I do?")
ask("What is the dress code on Fridays?")          # not in the docs → should refuse

❓ How much vacation do I get? (Acme-India)
💬 Based on the information provided in the passage, you would receive 18 days of paid annual leave each year at Acme-India. This can be seen in passage [2], which states "Employees accrue 18 days of paid leave annually."
📚 ['hr_policy_india.md › Sick leave', 'hr_policy_india.md › Annual leave', 'hr_policy_india.md › Remote work'] 

❓ How much vacation do I get? (Acme-US)
💬 Based on the information provided in the passage, you would typically receive 15 days of paid time off (PTO) annually as part of your annual leave policy at Acme-US. This amount does not carry over from one year to the next. [1]
📚 ['hr_policy_us.md › Sick leave', 'hr_policy_us.md › Annual leave', 'hr_policy_us.md › Remote work'] 

❓ I'm getting error E1047, what should I do? 
💬 I don't know based on the Acme documents.
📚 ['it_helpdesk.md › Error E1047: login token expired', 'it_helpdesk.md › Error E2001: disk quota exceeded', 'it_helpdesk.md › Password reset'] 

❓ What is th

### 📊 Output Significance
The final generation output successfully addresses the target queries using only the injected factual context and outputs precise citations (e.g., `[1]`). When asked about out-of-bounds topics, the model safely handles the query by refusing to answer rather than fabricating a response.

## ✅ Wrap-up
1. **BM25 matches words, embeddings match meaning.** Hybrid + RRF gets both.
2. **Bi-encoder retrieves, cross-encoder reranks.** Fast shortlist, then precise ordering.
3. **Chunks + metadata** make answers citable and filterable. Filter *before* searching.
4. **Match the metric:** normalise vectors so inner product = cosine.
5. **The LLM is the last step.** Retrieval quality decides the answer.


🔗 [Sentence-Transformers](https://sbert.net) · [FAISS wiki](https://github.com/facebookresearch/faiss/wiki) · [ChromaDB](https://docs.trychroma.com) · [rank_bm25](https://github.com/dorianbrown/rank_bm25) · [LangChain RAG tutorial](https://docs.langchain.com/oss/python/langchain/rag) · [RAG paper (Lewis et al., 2020)](https://arxiv.org/abs/2005.11401)